In [1]:
import os
import sys
import numpy as np

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_color_analysis import *
from syn_project.utils_notebook import *

%matplotlib widget

In [2]:
project_name = "syn"
condition = "base_model_0"
data = "biased_00"
switch_epoch = 0

checkpoint_epoch=0

n_samples = 32
show_results_fusion = False
fusion_attr_weight = 1.0
noise = 0.0

training_params = get_training_params(project_name, condition)
modules = get_setup_modules('syn', condition)
global_workspace = get_global_workspace("syn", condition, epoch=checkpoint_epoch, modules=modules)

/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/home/luc

In [3]:
training_params

{'experiment_name': 'base_model_0',
 'exclude_colors': True,
 'apply_custom_init': True,
 'config': Config(seed=0, max_train_size=500000, ood_seed=None, default_root_dir=PosixPath('checkpoints'), dataset=Dataset(path='/home/lucas/gwsyn/simple_shapes_dataset_biased_00'), training=Training(batch_size=2056, num_workers=16, devices=1, accelerator='gpu', fast_dev_run=False, max_steps=300000, enable_progress_bar=True, precision=32, float32_matmul_precision='highest', optim=Optim(lr=1e-05, max_lr=0.00015, start_lr=0.0001, end_lr=1e-05, pct_start=0.03, weight_decay=1e-06)), wandb=WanDB(enabled=True, save_dir='./wandb', project='Shimmer-SSD', entity='lexman-psl', reinit=False), logging=Logging(filter_images=['pred_trans_attr_to_attr', 'pred_trans_v_latents_to_v_latents', 'pred_trans_attr_to_v_latents', 'pred_trans_v_latents_to_attr', 'pred_cycle_v_latents_to_attr', 'pred_cycle_attr_to_v_latents', 'pred_cycle_v_latents_to_v_latents', 'pred_cycle_attr_to_attr'], log_train_medias_every_n_epochs=1,

In [4]:
training_params['config'].global_workspace.loss_coefficients

{'cycles': 1.0, 'contrastives': 1.0, 'demi_cycles': 1.0, 'translations': 1.0}

In [5]:
from deepdiff import DeepDiff
from pprint import pprint


condition = "ablation_base"
training_params2 = get_training_params(project_name, condition)
modules = get_setup_modules('syn', condition)
global_workspace2 = get_global_workspace("syn", condition, epoch=checkpoint_epoch, modules=modules)

training_params2 = get_training_params(project_name, condition)

diff = DeepDiff(training_params2, training_params, ignore_order=True)
pprint(diff)


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/home/luc

{'attribute_added': ["root['config'].default_root_dir._str"],
 'dictionary_item_added': ["root['swith_epoch']"],
 'dictionary_item_removed': ["root['attention_weight']"],
 'values_changed': {"root['config'].training.max_steps": {'new_value': 300000,
                                                          'old_value': 8000},
                    "root['experiment_name']": {'new_value': 'base_model_0',
                                                'old_value': 'ablation_base'}}}


In [6]:
global_workspace2.gw_mod.gw_encoders['v_latents']

GWEncoder(
  (0): Linear(in_features=12, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=256, bias=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): Linear(in_features=256, out_features=12, bias=True)
)

In [7]:
global_workspace.gw_mod.gw_encoders

ModuleDict(
  (attr): GWEncoder(
    (0): Linear(in_features=8, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=12, bias=True)
  )
  (v_latents): GWEncoder(
    (0): Linear(in_features=12, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=12, bias=True)
  )
  (color): GWEncoder(
    (0): Linear(in_features=3, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=12, bias=True)
  )
)

In [8]:
def are_models_equal(model1, model2):
    # On récupère les dictionnaires de poids
    sd1 = model1.state_dict()
    sd2 = model2.state_dict()

    # 1. Vérifier si les deux modèles ont le même nombre de paramètres/clés
    if sd1.keys() != sd2.keys():
        return False

    # 2. Comparer chaque tenseur un par un
    for key in sd1:
        if not torch.equal(sd1[key], sd2[key]):
            print(f"Différence détectée dans la couche : {key}")
            return False
            
    return True

# Utilisation
e1 = global_workspace2.gw_mod.gw_decoders['position']
e2 = global_workspace.gw_mod.gw_decoders['position']
equal = are_models_equal(e1, e2)

KeyError: 'position'

In [ ]:
equal

False